# 🧠 NLP Preprocessing Engine — Advanced
**Data Science Internship – February 2026**  
Task 1: Build a Robust NLP Preprocessing Engine

---

## 📦 Imports

In [1]:
import re
import string
from collections import Counter

---
## Task 1: Conceptual Understanding (Written)

### Q1. What is the difference between "Love" and "love" in NLP?

In NLP, text processing is case-sensitive by default. `"Love"` and `"love"` are treated as **two completely different tokens** unless we apply lowercasing. Most models and vocabulary lookups would not recognize them as the same word. This can hurt model performance — for example, a sentiment classifier trained mostly on lowercase text would fail to recognize `"Love"` as a positive word. That is why lowercasing is one of the very first preprocessing steps in most NLP pipelines.

---

### Q2. What happens if stopwords are not removed?

If stopwords (words like `"the"`, `"is"`, `"a"`, `"and"`) are not removed:
- They **dominate the token frequency** counts, drowning out meaningful words.
- Models like TF-IDF become less effective because these words appear everywhere and carry no topic-specific weight.
- Vocabulary size increases unnecessarily, making training slower and memory-heavy.
- For tasks like topic modelling or keyword extraction, results become noisy and less interpretable.

---

### Q3. Two real-world scenarios where removing stopwords can be HARMFUL

1. **Sentiment Analysis** — The phrase `"not good"` is negative, but removing the stopword `"not"` turns it into `"good"` — completely reversing the sentiment. Words like `"no"`, `"not"`, `"never"` are critical for negation and must be preserved.

2. **Question Answering / Chatbots** — For a query like `"Who is the president of India?"`, removing stopwords gives `"president India"`. This loses the grammatical structure needed for accurate answer retrieval or intent detection.

---

### Q4. Difference between Stemming and Lemmatization

| | Stemming | Lemmatization |
|---|---|---|
| **What it does** | Chops off word endings using crude rules | Reduces word to its dictionary base form (lemma) |
| **Linguistic accuracy** | Low — often produces non-words | High — always produces a valid word |
| **Speed** | Faster | Slower (needs a vocabulary/grammar lookup) |
| **Example** | `"running"` → `"run"`, `"studies"` → `"studi"` | `"running"` → `"run"`, `"studies"` → `"study"` |
| **Best for** | Search engines, quick text mining | High-accuracy NLP tasks like chatbots, QA |

**Summary:** Stemming is fast but rough; lemmatization is slower but linguistically correct.

---
## Task 2: Build Advanced Preprocessing Function

In [2]:
# Words that are short (≤2 chars) but must be kept because they carry meaning
MEANINGFUL_SHORT_WORDS = {"no", "not"}

def preprocess_text(text):
    """
    Advanced NLP preprocessing function.

    Steps applied (in order):
      1. Handle edge cases (None, empty, non-string)
      2. Remove emojis and non-ASCII characters
      3. Remove URLs and email-like patterns
      4. Convert to lowercase
      5. Handle repeated characters (e.g. 'loooove' → 'love')
      6. Remove punctuation
      7. Remove numbers
      8. Tokenize (split on whitespace)
      9. Remove extra spaces / empty tokens
     10. Remove short tokens (len ≤ 2) unless in MEANINGFUL_SHORT_WORDS

    Args:
        text (str): Raw input text

    Returns:
        list[str]: List of clean tokens
    """
    # ── Step 1: Edge-case handling (Task 7)
    if not isinstance(text, str):
        return []
    text = text.strip()
    if not text:
        return []

    # ── Step 2: Remove emojis and non-ASCII characters
    # This also strips emoji-only strings entirely
    text = text.encode("ascii", errors="ignore").decode("ascii")

    # ── Step 3: Remove URLs (http/https/www) and email-like patterns
    text = re.sub(r"http\S+|https\S+|www\.\S+", "", text)   # URLs
    text = re.sub(r"\S+@\S+\.\S+", "", text)                 # emails

    # ── Step 4: Convert to lowercase
    text = text.lower()

    # ── Step 5: Collapse repeated characters
    # "loooove" → "love",  "sooo" → "so",  "!!!!!" → "!"
    # We keep at most 2 of any repeated character, then strip punctuation next
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)
    # A second pass collapses "\1\1" for non-alpha (like "!!") — optional cleanup
    # For letters like "ll" we do NOT collapse further ("well", "call" are valid)
    # but for obvious slang like "sooo" one pass is sufficient to get "soo" → still fine

    # ── Step 6: Remove punctuation (and symbols like $, %, /)
    text = re.sub(r"[^\w\s]", "", text)   # keep word chars and spaces only

    # ── Step 7: Remove numbers (standalone digits)
    text = re.sub(r"\b\d+\b", "", text)

    # ── Step 8: Tokenize on whitespace
    tokens = text.split()

    # ── Step 9 & 10: Filter tokens
    clean_tokens = [
        t for t in tokens
        if len(t) > 2 or t in MEANINGFUL_SHORT_WORDS
    ]

    return clean_tokens


# ── Quick sanity checks
print(preprocess_text("I have 2 dogs"))                    # ['have', 'dogs']
print(preprocess_text("This  is  good"))                   # ['this', 'good']
print(preprocess_text("soooo goooood!!!"))                 # ['soo', 'good']  (2-char collapse)
print(preprocess_text("WOW!!! This is GREAT!!!"))          # ['wow', 'this', 'great']
print(preprocess_text("Visit http://example.com now"))     # ['visit', 'now']
print(preprocess_text("I am not happy"))                   # ['not', 'happy']  ('not' kept)
print(preprocess_text(""))                                  # []
print(preprocess_text("😍😍😍"))                             # []
print(preprocess_text("123456"))                            # []

['have', 'dogs']
['this', 'good']
['soo', 'good']
['wow', 'this', 'great']
['visit', 'now']
['not', 'happy']
[]
[]
[]


---
## Task 3: Stress Testing — 10 Diverse Sentences

In [3]:
test_sentences = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product 😍😍",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this",
]

print("=" * 70)
print(f"{'STRESS TEST RESULTS':^70}")
print("=" * 70)

results = []   # store for later tasks

for i, sentence in enumerate(test_sentences, 1):
    tokens = preprocess_text(sentence)
    clean_sentence = " ".join(tokens)
    results.append({
        "original":       sentence,
        "tokens":         tokens,
        "clean_sentence": clean_sentence,
    })
    print(f"\n[{i}] Original  : {sentence}")
    print(f"    Tokens    : {tokens}")
    print(f"    Cleaned   : {clean_sentence}")

print("\n" + "=" * 70)

                         STRESS TEST RESULTS                          

[1] Original  : Get 100% FREE access now!!!
    Tokens    : ['get', 'free', 'access', 'now']
    Cleaned   : get free access now

[2] Original  : I absolutely looooved this product 😍😍
    Tokens    : ['absolutely', 'looved', 'this', 'product']
    Cleaned   : absolutely looved this product

[3] Original  : Worst service ever... 0/10
    Tokens    : ['worst', 'service', 'ever']
    Cleaned   : worst service ever

[4] Original  : Call me at 9876543210
    Tokens    : ['call']
    Cleaned   : call

[5] Original  : This is THE best course!!!
    Tokens    : ['this', 'the', 'best', 'course']
    Cleaned   : this the best course

[6] Original  : Visit https://openai.com now!
    Tokens    : ['visit', 'now']
    Cleaned   : visit now

[7] Original  : Nooooo this is baaad!!!
    Tokens    : ['noo', 'this', 'baad']
    Cleaned   : noo this baad

[8] Original  : OK OK OK I got it
    Tokens    : ['got']
    Cleaned   : got



---
## Task 4: Token Analytics

In [4]:
print("=" * 70)
print(f"{'TOKEN ANALYTICS':^70}")
print("=" * 70)
print(f"{'#':<4} {'Total':>7} {'Unique':>7} {'Avg Len':>9}   Original")
print("-" * 70)

analytics = []

for i, r in enumerate(results, 1):
    tokens = r["tokens"]
    total   = len(tokens)
    unique  = len(set(tokens))
    avg_len = round(sum(len(t) for t in tokens) / total, 2) if total else 0
    analytics.append({"total": total, "unique": unique, "avg_len": avg_len})
    print(f"{i:<4} {total:>7} {unique:>7} {avg_len:>9}   {r['original'][:45]}")

# ── Analysis Questions
totals   = [a["total"] for a in analytics]
original_token_counts = [len(s.split()) for s in test_sentences]
noise    = [original_token_counts[i] - analytics[i]["total"] for i in range(len(analytics))]

most_noise_idx    = noise.index(max(noise))
most_retained_idx = totals.index(max(totals))

print(f"\n📊 Analysis Questions")
print(f"  Most noisy sentence      : [{most_noise_idx+1}] \"{test_sentences[most_noise_idx]}\"")
print(f"  Most tokens retained     : [{most_retained_idx+1}] \"{test_sentences[most_retained_idx]}\"")

                           TOKEN ANALYTICS                            
#      Total  Unique   Avg Len   Original
----------------------------------------------------------------------
1          4       4       4.0   Get 100% FREE access now!!!
2          4       4      6.75   I absolutely looooved this product 😍😍
3          3       3      5.33   Worst service ever... 0/10
4          1       1       4.0   Call me at 9876543210
5          4       4      4.25   This is THE best course!!!
6          2       2       4.0   Visit https://openai.com now!
7          3       3      3.67   Nooooo this is baaad!!!
8          1       1       3.0   OK OK OK I got it
9          4       4       4.5   Win $$$ now!!! Limited offer!!!
10         4       4       4.0   I am not happy with this

📊 Analysis Questions
  Most noisy sentence      : [8] "OK OK OK I got it"
  Most tokens retained     : [1] "Get 100% FREE access now!!!"


---
## Task 5: Frequency Analysis

In [5]:
# Combine all tokens from all sentences
all_tokens = [token for r in results for token in r["tokens"]]

freq = Counter(all_tokens)

print("=" * 50)
print("Top 10 Most Frequent Words")
print("=" * 50)
for word, count in freq.most_common(10):
    print(f"  {word:<20} : {count}")

print()
print("=" * 50)
print("Top 5 Least Frequent Words")
print("=" * 50)
# Words that appear fewest times (excluding ties at 1 to pick meaningful ones)
least_common = freq.most_common()[-5:]
for word, count in least_common:
    print(f"  {word:<20} : {count}")

print(f"\nTotal unique tokens across all sentences : {len(freq)}")
print(f"Total tokens (with repetition)           : {len(all_tokens)}")

Top 10 Most Frequent Words
  this                 : 4
  now                  : 3
  get                  : 1
  free                 : 1
  access               : 1
  absolutely           : 1
  looved               : 1
  product              : 1
  worst                : 1
  service              : 1

Top 5 Least Frequent Words
  limited              : 1
  offer                : 1
  not                  : 1
  happy                : 1
  with                 : 1

Total unique tokens across all sentences : 25
Total tokens (with repetition)           : 30


---
## Task 6: Build Full Pipeline

In [6]:
def full_pipeline(text_list):
    """
    Full NLP preprocessing pipeline for a list of texts.

    Args:
        text_list (list[str]): List of raw input strings

    Returns:
        dict: {
            'tokens'         : list of token lists (one per sentence),
            'clean_sentences': list of cleaned sentence strings
        }
    """
    if not isinstance(text_list, list):
        raise TypeError(f"Expected a list, got {type(text_list).__name__}")

    all_tokens       = []
    clean_sentences  = []

    for text in text_list:
        tokens = preprocess_text(text)          # uses our Task 2 function
        all_tokens.append(tokens)
        clean_sentences.append(" ".join(tokens))

    return {
        "tokens":          all_tokens,
        "clean_sentences": clean_sentences,
    }


# ── Run the pipeline
output = full_pipeline(test_sentences)

print("Full Pipeline Output")
print("=" * 60)
for i, (tokens, sentence) in enumerate(
        zip(output["tokens"], output["clean_sentences"]), 1):
    print(f"[{i}] tokens          : {tokens}")
    print(f"    clean_sentence  : {sentence}")
    print()

Full Pipeline Output
[1] tokens          : ['get', 'free', 'access', 'now']
    clean_sentence  : get free access now

[2] tokens          : ['absolutely', 'looved', 'this', 'product']
    clean_sentence  : absolutely looved this product

[3] tokens          : ['worst', 'service', 'ever']
    clean_sentence  : worst service ever

[4] tokens          : ['call']
    clean_sentence  : call

[5] tokens          : ['this', 'the', 'best', 'course']
    clean_sentence  : this the best course

[6] tokens          : ['visit', 'now']
    clean_sentence  : visit now

[7] tokens          : ['noo', 'this', 'baad']
    clean_sentence  : noo this baad

[8] tokens          : ['got']
    clean_sentence  : got

[9] tokens          : ['win', 'now', 'limited', 'offer']
    clean_sentence  : win now limited offer

[10] tokens          : ['not', 'happy', 'with', 'this']
    clean_sentence  : not happy with this



---
## Task 7: Error Handling

In [7]:
# Edge cases — all handled gracefully inside preprocess_text()
edge_cases = {
    "Empty string"    : "",
    "Only emojis"     : "😍🔥💯🎉",
    "Only numbers"    : "1234567890",
    "None value"      : None,
    "Only spaces"     : "     ",
    "Only punctuation": "!!! ??? ...",
}

print("=" * 60)
print(f"{'ERROR HANDLING TEST CASES':^60}")
print("=" * 60)

for label, value in edge_cases.items():
    result = preprocess_text(value)
    print(f"Input  : {repr(value):<30}  ({label})")
    print(f"Output : {result}")
    print("-" * 60)

# Pipeline error handling
print("\nPipeline edge cases:")
edge_list = ["", "😍😍", "123456"]
pipeline_output = full_pipeline(edge_list)
for i, (t, s) in enumerate(zip(pipeline_output["tokens"],
                                pipeline_output["clean_sentences"]), 1):
    print(f"  [{i}] tokens: {t}  |  clean: '{s}'")

print("\n✅ All edge cases handled without errors.")

                 ERROR HANDLING TEST CASES                  
Input  : ''                              (Empty string)
Output : []
------------------------------------------------------------
Input  : '😍🔥💯🎉'                          (Only emojis)
Output : []
------------------------------------------------------------
Input  : '1234567890'                    (Only numbers)
Output : []
------------------------------------------------------------
Input  : None                            (None value)
Output : []
------------------------------------------------------------
Input  : '     '                         (Only spaces)
Output : []
------------------------------------------------------------
Input  : '!!! ??? ...'                   (Only punctuation)
Output : []
------------------------------------------------------------

Pipeline edge cases:
  [1] tokens: []  |  clean: ''
  [2] tokens: []  |  clean: ''
  [3] tokens: []  |  clean: ''

✅ All edge cases handled without errors.


---
## ✅ Summary

| Task | Description | Status |
|------|-------------|--------|
| Task 1 | Conceptual questions answered | ✅ |
| Task 2 | `preprocess_text()` with all required features | ✅ |
| Task 3 | Stress-tested on 10 diverse sentences | ✅ |
| Task 4 | Token analytics (total, unique, avg length) | ✅ |
| Task 5 | Frequency analysis — top 10 & bottom 5 | ✅ |
| Task 6 | `full_pipeline()` returning tokens + clean sentences | ✅ |
| Task 7 | Edge cases: empty, emoji-only, number-only, None | ✅ |